# Retrieval Results Analysis

This notebook uses `tasting_analysis.py` workflows to:

1. Run CCF robustness checks between observed combined spectra and retrieval scaled model flux.
2. Compute chemistry diagnostics (C/O and `12CO/13CO`) with 1-sigma and 3-sigma intervals.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from tasting_analysis import run_ccf_workflow, run_chemistry_workflow, run_equil_chemistry_workflow

## Retrieval: 1992770_N700_ev0.5_Normsavgol_PerChipScaleFalse

Free chemistry (log_H2O, log_12CO, log_13CO, log_CH4), two-night fit.  
Chemistry diagnostics: C/O, [C/H] (×solar, dex), 12CO/13CO.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from analysis import (
    plot_ccf, plot_free_chemistry,
    plot_binned_model_vs_data, plot_pt_profile,
    _FREE_CHEM_PARAMS,
)
from tasting_analysis import run_ccf_workflow, run_free_chemistry_workflow

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------
workpath     = Path('/data2/peng')
retrieval_id = '1992770_N700_ev0.5_Normsavgol_PerChipScaleFalse'
night        = '2022-12-31'

print('workpath:', workpath)
print('retrieval_id:', retrieval_id)
print('night:', night)
print('retrieval exists:', (workpath / 'retrievals' / retrieval_id).exists())
# ------------------------------------------------------------------
# 1) CCF robustness workflow
# ------------------------------------------------------------------
ccf_out = run_ccf_workflow(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night=night,
    rvlag=np.arange(-10, 10, 0.1),
    clean_grids=((-10, -1), (0, 10)),
    n_shuffle=200,
    random_seed=42,
    combined_spectrum_name="extracted_spectra_combined_sigmaclipper.npy",
)

print('Combined spectra file:', ccf_out['combined_path'])
print('Valid pixels used:', ccf_out['valid_pixels'])
print(f"Peak SNR = {ccf_out['peak_snr']:.2f} at RV = {ccf_out['peak_rv']:.1f} km/s")
print(f"Null peak SNR mean\u00b1std = {ccf_out['null_mu']:.2f} \u00b1 {ccf_out['null_sigma']:.2f}")
print(f"Detection z-score (vs shuffled null) = {ccf_out['z_score']:.2f}")

plot_ccf(ccf_out, retrieval_id)
# ------------------------------------------------------------------
# 2) Free-chemistry posterior diagnostics
# ------------------------------------------------------------------
chem_out = run_free_chemistry_workflow(workpath=workpath, retrieval_id=retrieval_id)

print('Posterior shape:', chem_out['posterior'].shape)
print('Retrieved params:', chem_out['columns'])
display(chem_out['summary'])

print('\nText summary:')
for _, row in chem_out['summary'].iterrows():
    print(
        f"{row['parameter']}: median={row['p50']:.4g}, "
        f"1\u03c3=[-{row['minus_1sigma']:.3g}, +{row['plus_1sigma']:.3g}], "
        f"3\u03c3=[-{row['minus_3sigma']:.3g}, +{row['plus_3sigma']:.3g}]"
    )

plot_free_chemistry(chem_out, retrieval_id)


In [ ]:
# ------------------------------------------------------------------
# Night 1 (2022-12-31): binned data vs model, per-order-detector
# ------------------------------------------------------------------
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)


In [ ]:
# ------------------------------------------------------------------
# Night 2 (2023-01-01): model_flux_N2 vs data — binned, per-order-detector
# ------------------------------------------------------------------
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2023-01-01',
    model_wave_npy='retrieval_model_wave_N2.npy',
    model_flux_npy='retrieval_model_flux_N2.npy',
)


In [ ]:
# ------------------------------------------------------------------
# P-T profile posterior — 1sigma / 2sigma / 3sigma bands
# ------------------------------------------------------------------
plot_pt_profile(
    workpath=workpath,
    retrieval_id=retrieval_id,
    param_keys=_FREE_CHEM_PARAMS,
)


## Retrieval: 1993932_N700_ev0.5_Normsavgol_PerChipScaleFalse

Equilibrium chemistry (C_H, C/O, log_12CO_13CO), two-night fit.  
Chemistry diagnostics: C/O, [C/H] (dex), 12CO/13CO.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from analysis import (
    plot_ccf, plot_equil_chemistry,
    plot_binned_model_vs_data, plot_pt_profile,
    _EQUIL_CHEM_PARAMS,
)
from tasting_analysis import run_ccf_workflow, run_equil_chemistry_workflow

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------
workpath     = Path('/data2/peng')
retrieval_id = '1993932_N700_ev0.5_Normsavgol_PerChipScaleFalse'
night        = '2022-12-31'

print('workpath:', workpath)
print('retrieval_id:', retrieval_id)
print('night:', night)
print('retrieval exists:', (workpath / 'retrievals' / retrieval_id).exists())
# ------------------------------------------------------------------
# 1) CCF robustness workflow
# ------------------------------------------------------------------
ccf_out = run_ccf_workflow(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night=night,
    rvlag=np.arange(-10, 10, 0.1),
    clean_grids=((-10, -1), (0, 10)),
    n_shuffle=200,
    random_seed=42,
    combined_spectrum_name="extracted_spectra_combined_sigmaclipper.npy",
)

print('Combined spectra file:', ccf_out['combined_path'])
print('Valid pixels used:', ccf_out['valid_pixels'])
print(f"Peak SNR = {ccf_out['peak_snr']:.2f} at RV = {ccf_out['peak_rv']:.1f} km/s")
print(f"Null peak SNR mean\u00b1std = {ccf_out['null_mu']:.2f} \u00b1 {ccf_out['null_sigma']:.2f}")
print(f"Detection z-score (vs shuffled null) = {ccf_out['z_score']:.2f}")

plot_ccf(ccf_out, retrieval_id)
# ------------------------------------------------------------------
# 2) Equilibrium-chemistry posterior diagnostics
# ------------------------------------------------------------------
chem_out = run_equil_chemistry_workflow(workpath=workpath, retrieval_id=retrieval_id)

print('Posterior shape:', chem_out['posterior'].shape)
print('Retrieved params:', chem_out['columns'])
display(chem_out['summary'])

print('\nText summary:')
for _, row in chem_out['summary'].iterrows():
    print(
        f"{row['parameter']}: median={row['p50']:.4g}, "
        f"1\u03c3=[-{row['minus_1sigma']:.3g}, +{row['plus_1sigma']:.3g}], "
        f"3\u03c3=[-{row['minus_3sigma']:.3g}, +{row['plus_3sigma']:.3g}]"
    )

plot_equil_chemistry(chem_out, retrieval_id)


In [ ]:
# ------------------------------------------------------------------
# Night 1 (2022-12-31): binned data vs model, per-order-detector
# ------------------------------------------------------------------
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)


In [ ]:
# ------------------------------------------------------------------
# Night 2 (2023-01-01): model_flux_N2 vs data — binned, per-order-detector
# ------------------------------------------------------------------
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2023-01-01',
    model_wave_npy='retrieval_model_wave_N2.npy',
    model_flux_npy='retrieval_model_flux_N2.npy',
)


In [ ]:
# ------------------------------------------------------------------
# P-T profile posterior — 1sigma / 2sigma / 3sigma bands
# ------------------------------------------------------------------
plot_pt_profile(
    workpath=workpath,
    retrieval_id=retrieval_id,
    param_keys=_EQUIL_CHEM_PARAMS,
)


## Retrieval: 2399524_N700_ev0.5_Normsavgol_PerChipScaleFalse

**Series 002PM** — Piette & Madhusudhan (2020) P-T Profile  
Free chemistry (log_H2O, log_12CO, log_13CO, log_CH4), two-night fit.  
P-T: T_anchor at 1 bar + dT_1…dT_7 (PCHIP monotonic spline, no Gaussian smoothing).  
Chemistry diagnostics: C/O, [C/H] (×solar, dex), ¹²CO/¹³CO.  

**Source**: `Guidebook_GAStronomy_Piette_v1.0.py` | N_live=700, ev_tol=0.5, norm=savgol, per_chip_scale=False

In [ ]:
import sys
sys.path.insert(0, '/data2/peng/Recipe_DH_Tau_B')  # ensure Recipe_DH_Tau_B tasting_analysis is used

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from analysis import (
    plot_ccf, plot_free_chemistry,
    plot_binned_model_vs_data,
)
from tasting_analysis import run_ccf_workflow, run_free_chemistry_workflow
from scipy.interpolate import PchipInterpolator

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------
workpath     = Path('/data2/peng')
retrieval_id = '2399524_N700_ev0.5_Normsavgol_PerChipScaleFalse'
night        = '2022-12-31'

print('workpath:', workpath)
print('retrieval_id:', retrieval_id)
print('night:', night)
print('retrieval exists:', (workpath / 'retrievals' / retrieval_id).exists())
# ------------------------------------------------------------------
# 1) CCF robustness workflow
# ------------------------------------------------------------------
ccf_out = run_ccf_workflow(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night=night,
    rvlag=np.arange(-10, 10, 0.1),
    clean_grids=((-10, -1), (0, 10)),
    n_shuffle=200,
    random_seed=42,
    combined_spectrum_name="extracted_spectra_combined_sigmaclipper.npy",
)

print('Combined spectra file:', ccf_out['combined_path'])
print('Valid pixels used:', ccf_out['valid_pixels'])
print(f"Peak SNR = {ccf_out['peak_snr']:.2f} at RV = {ccf_out['peak_rv']:.1f} km/s")
print(f"Null peak SNR mean±std = {ccf_out['null_mu']:.2f} ± {ccf_out['null_sigma']:.2f}")
print(f"Detection z-score (vs shuffled null) = {ccf_out['z_score']:.2f}")

plot_ccf(ccf_out, retrieval_id)
# ------------------------------------------------------------------
# 2) Free-chemistry posterior diagnostics
# ------------------------------------------------------------------
chem_out = run_free_chemistry_workflow(workpath=workpath, retrieval_id=retrieval_id)

print('Posterior shape:', chem_out['posterior'].shape)
print('Retrieved params:', chem_out['columns'])
display(chem_out['summary'])

print('\nText summary:')
for _, row in chem_out['summary'].iterrows():
    print(
        f"{row['parameter']}: median={row['p50']:.4g}, "
        f"1σ=[-{row['minus_1sigma']:.3g}, +{row['plus_1sigma']:.3g}], "
        f"3σ=[-{row['minus_3sigma']:.3g}, +{row['plus_3sigma']:.3g}]"
    )

plot_free_chemistry(chem_out, retrieval_id)


In [ ]:
# ------------------------------------------------------------------
# Night 1 (2022-12-31): binned data vs model, per-order-detector
# ------------------------------------------------------------------
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)


In [ ]:
# ------------------------------------------------------------------
# Night 2 (2023-01-01): model_flux_N2 vs data — binned, per-order-detector
# ------------------------------------------------------------------
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2023-01-01',
    model_wave_npy='retrieval_model_wave_N2.npy',
    model_flux_npy='retrieval_model_flux_N2.npy',
)


In [ ]:
# ------------------------------------------------------------------
# P-T profile posterior — 1σ / 2σ / 3σ bands
# Piette+2020 parameterisation (T_anchor + dT_1…dT_7, PCHIP spline)
# analysis.plot_pt_profile() is DG-only; computed inline here.
# ------------------------------------------------------------------
import pickle

_PIETTE_FREE_CHEM_PARAMS = [
    "rv_N1", "rv_N2", "vsini", "epsilon", "log_g",
    "T_anchor", "dT_1", "dT_2", "dT_3", "dT_4", "dT_5", "dT_6", "dT_7",
    "log_H2O", "log_12CO", "log_13CO", "log_CH4",
]

LOG_P_NODES = np.array([2.0, 1.5, 1.0, 0.5, 0.0, -1.0, -2.0, -3.0])
ANCHOR_IDX  = 4   # log P = 0.0 (1 bar)
pressure    = np.logspace(-5, 2, 50)

retrieval_dir = workpath / "retrievals" / retrieval_id
posterior = np.load(retrieval_dir / "final_posterior.npy")
col = {k: i for i, k in enumerate(_PIETTE_FREE_CHEM_PARAMS)}

n_samples = posterior.shape[0]
all_temps = np.empty((n_samples, len(pressure)))

for i, s in enumerate(posterior):
    T_anchor = s[col["T_anchor"]]
    dT = np.array([s[col[f"dT_{j+1}"]] for j in range(7)])

    T_nodes = np.empty(8)
    T_nodes[ANCHOR_IDX] = T_anchor
    for j in range(ANCHOR_IDX - 1, -1, -1):
        T_nodes[j] = T_nodes[j + 1] + dT[j]
    for j in range(ANCHOR_IDX + 1, 8):
        T_nodes[j] = T_nodes[j - 1] - dT[j - 1]

    log_P_asc = LOG_P_NODES[::-1]
    T_asc     = T_nodes[::-1]
    pchip     = PchipInterpolator(log_P_asc, T_asc)
    log_P_atm = np.log10(pressure)
    temp      = pchip(log_P_atm)
    temp      = np.where(log_P_atm < log_P_asc[0],  T_asc[0],  temp)
    temp      = np.where(log_P_atm > log_P_asc[-1], T_asc[-1], temp)
    all_temps[i] = np.clip(temp, 1.0, 30000.0)

pcts = np.percentile(all_temps, [0.27, 2.28, 15.87, 50.0, 84.13, 97.72, 99.73], axis=0)

fig, ax = plt.subplots(figsize=(5, 7))
color = "steelblue"
ax.fill_betweenx(pressure, pcts[0], pcts[6], color=color, alpha=0.15,
                 linewidth=0, label="3σ (99.7%)")
ax.fill_betweenx(pressure, pcts[1], pcts[5], color=color, alpha=0.30,
                 linewidth=0, label="2σ (95.4%)")
ax.fill_betweenx(pressure, pcts[2], pcts[4], color=color, alpha=0.55,
                 linewidth=0, label="1σ (68.3%)")
ax.plot(pcts[3], pressure, color="navy", lw=1.8, label="Median")

# Mark the 8 fixed P-T anchor nodes
for logp in LOG_P_NODES:
    ax.axhline(10**logp, color="gray", lw=0.5, ls=":", alpha=0.6)

ax.set_ylim(1e-5, 10)
ax.set_xlim(0, 3000)
ax.set_yscale("log")
ax.invert_yaxis()
ax.set_xlabel("Temperature (K)", fontsize=11)
ax.set_ylabel("Pressure (bar)", fontsize=11)
ax.set_title(
    f"P-T profile posterior (002PM, Piette+2020 PCHIP)\n{retrieval_id}",
    fontsize=9,
)
ax.legend(fontsize=9, loc="upper right")
ax.grid(alpha=0.2, lw=0.5)
ax.tick_params(labelsize=9)
plt.tight_layout()
plt.show()

print(f"Posterior samples used: {n_samples}")
print(f"Median T range: {pcts[3].min():.0f} – {pcts[3].max():.0f} K")
print(f"T_anchor median: {np.percentile(posterior[:, col['T_anchor']], 50):.1f} K")


## Retrieval: 885739_N700_ev0.5_Normsavgol_PerChipScaleFalse

**Series 002PM-EQ** — Piette & Madhusudhan (2020) P-T Profile  
Equilibrium chemistry (C_H, C/O, log_12CO_13CO), two-night fit.  
P-T: T_anchor at 1 bar + dT_1…dT_7 (PCHIP monotonic spline, no Gaussian smoothing).  
T_anchor prior: U(1000, 3000) K (widened from Guidebook default [1500, 3000]).  
Chemistry diagnostics: C/O, [C/H] (×solar, dex), ¹²CO/¹³CO.  

**Source**: `Guidebook_GAStronomy_Piette_v1.0.py` | N_live=700, ev_tol=0.5, norm=savgol, per_chip_scale=False

In [ ]:
import sys
sys.path.insert(0, '/data2/peng/Recipe_DH_Tau_B')  # ensure Recipe_DH_Tau_B tasting_analysis is used

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from analysis import (
    plot_ccf, plot_equil_chemistry,
    plot_binned_model_vs_data,
)
from tasting_analysis import run_ccf_workflow, run_equil_chemistry_workflow
from scipy.interpolate import PchipInterpolator

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------
workpath     = Path('/data2/peng')
retrieval_id = '885739_N700_ev0.5_Normsavgol_PerChipScaleFalse'
night        = '2022-12-31'

print('workpath:', workpath)
print('retrieval_id:', retrieval_id)
print('night:', night)
print('retrieval exists:', (workpath / 'retrievals' / retrieval_id).exists())
# ------------------------------------------------------------------
# 1) CCF robustness workflow
# ------------------------------------------------------------------
ccf_out = run_ccf_workflow(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night=night,
    rvlag=np.arange(-10, 10, 0.1),
    clean_grids=((-10, -1), (0, 10)),
    n_shuffle=200,
    random_seed=42,
    combined_spectrum_name="extracted_spectra_combined_sigmaclipper.npy",
)

print('Combined spectra file:', ccf_out['combined_path'])
print('Valid pixels used:', ccf_out['valid_pixels'])
print(f"Peak SNR = {ccf_out['peak_snr']:.2f} at RV = {ccf_out['peak_rv']:.1f} km/s")
print(f"Null peak SNR mean\u00b1std = {ccf_out['null_mu']:.2f} \u00b1 {ccf_out['null_sigma']:.2f}")
print(f"Detection z-score (vs shuffled null) = {ccf_out['z_score']:.2f}")

plot_ccf(ccf_out, retrieval_id)
# ------------------------------------------------------------------
# 2) Equilibrium-chemistry posterior diagnostics
# ------------------------------------------------------------------
chem_out = run_equil_chemistry_workflow(workpath=workpath, retrieval_id=retrieval_id)

print('Posterior shape:', chem_out['posterior'].shape)
print('Retrieved params:', chem_out['columns'])
display(chem_out['summary'])

print('\nText summary:')
for _, row in chem_out['summary'].iterrows():
    print(
        f"{row['parameter']}: median={row['p50']:.4g}, "
        f"1\u03c3=[-{row['minus_1sigma']:.3g}, +{row['plus_1sigma']:.3g}], "
        f"3\u03c3=[-{row['minus_3sigma']:.3g}, +{row['plus_3sigma']:.3g}]"
    )

plot_equil_chemistry(chem_out, retrieval_id)


In [ ]:
# ------------------------------------------------------------------
# Night 1 (2022-12-31): binned data vs model, per-order-detector
# ------------------------------------------------------------------
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)


In [ ]:
# ------------------------------------------------------------------
# Night 2 (2023-01-01): model_flux_N2 vs data — binned, per-order-detector
# ------------------------------------------------------------------
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2023-01-01',
    model_wave_npy='retrieval_model_wave_N2.npy',
    model_flux_npy='retrieval_model_flux_N2.npy',
)


In [ ]:
# ------------------------------------------------------------------
# P-T profile posterior — 1σ / 2σ / 3σ bands
# Piette+2020 parameterisation (T_anchor + dT_1…dT_7, PCHIP spline)
# analysis.plot_pt_profile() is DG-only; computed inline here.
# ------------------------------------------------------------------

_PIETTE_EQUIL_CHEM_PARAMS = [
    "rv_N1", "rv_N2", "vsini", "epsilon", "log_g",
    "T_anchor", "dT_1", "dT_2", "dT_3", "dT_4", "dT_5", "dT_6", "dT_7",
    "C_H", "C/O", "log_12CO_13CO",
]

LOG_P_NODES = np.array([2.0, 1.5, 1.0, 0.5, 0.0, -1.0, -2.0, -3.0])
ANCHOR_IDX  = 4   # log P = 0.0 (1 bar)
pressure    = np.logspace(-5, 2, 50)

retrieval_dir = workpath / 'retrievals' / retrieval_id
posterior = np.load(retrieval_dir / 'final_posterior.npy')
col = {k: i for i, k in enumerate(_PIETTE_EQUIL_CHEM_PARAMS)}

n_samples = posterior.shape[0]
all_temps = np.empty((n_samples, len(pressure)))

for i, s in enumerate(posterior):
    T_anchor = s[col['T_anchor']]
    dT = np.array([s[col[f'dT_{j+1}']] for j in range(7)])

    T_nodes = np.empty(8)
    T_nodes[ANCHOR_IDX] = T_anchor
    for j in range(ANCHOR_IDX - 1, -1, -1):
        T_nodes[j] = T_nodes[j + 1] + dT[j]
    for j in range(ANCHOR_IDX + 1, 8):
        T_nodes[j] = T_nodes[j - 1] - dT[j - 1]

    log_P_asc = LOG_P_NODES[::-1]
    T_asc     = T_nodes[::-1]
    pchip     = PchipInterpolator(log_P_asc, T_asc)
    log_P_atm = np.log10(pressure)
    temp      = pchip(log_P_atm)
    temp      = np.where(log_P_atm < log_P_asc[0],  T_asc[0],  temp)
    temp      = np.where(log_P_atm > log_P_asc[-1], T_asc[-1], temp)
    all_temps[i] = np.clip(temp, 1.0, 30000.0)

pcts = np.percentile(all_temps, [0.27, 2.28, 15.87, 50.0, 84.13, 97.72, 99.73], axis=0)

fig, ax = plt.subplots(figsize=(5, 7))
color = 'steelblue'
ax.fill_betweenx(pressure, pcts[0], pcts[6], color=color, alpha=0.15,
                 linewidth=0, label='3σ (99.7%)')
ax.fill_betweenx(pressure, pcts[1], pcts[5], color=color, alpha=0.30,
                 linewidth=0, label='2σ (95.4%)')
ax.fill_betweenx(pressure, pcts[2], pcts[4], color=color, alpha=0.55,
                 linewidth=0, label='1σ (68.3%)')
ax.plot(pcts[3], pressure, color='navy', lw=1.8, label='Median')

# Mark the 8 fixed P-T anchor nodes
for logp in LOG_P_NODES:
    ax.axhline(10**logp, color='gray', lw=0.5, ls=':', alpha=0.6)

ax.set_ylim(1e-5, 10)
ax.set_xlim(1500, 3000)
ax.set_yscale('log')
ax.invert_yaxis()
ax.set_xlabel('Temperature (K)', fontsize=11)
ax.set_ylabel('Pressure (bar)', fontsize=11)
ax.set_title(
    f'P-T profile posterior (002PM-EQ, Piette+2020 PCHIP)\n{retrieval_id}',
    fontsize=9,
)
ax.legend(fontsize=9, loc='upper right')
ax.grid(alpha=0.2, lw=0.5)
ax.tick_params(labelsize=9)
plt.tight_layout()
plt.show()

print(f'Posterior samples used: {n_samples}')
print(f'Median T range: {pcts[3].min():.0f} – {pcts[3].max():.0f} K')
print(f'T_anchor median: {np.percentile(posterior[:, col["T_anchor"]], 50):.1f} K')
print(f'C_H median:      {np.percentile(posterior[:, col["C_H"]], 50):.3f} dex')
print(f'C/O median:      {np.percentile(posterior[:, col["C/O"]], 50):.3f}')
print(f'log_12CO_13CO median: {np.percentile(posterior[:, col["log_12CO_13CO"]], 50):.3f}')


## Retrieval: 936677_N700_ev0.5_Normsavgol_PerChipScaleFalse

**Series 003PM-EQ** — Piette & Madhusudhan (2020) P-T Profile  
Equilibrium chemistry (C_H, C/O, log_12CO_13CO), two-night fit.  
P-T: T_anchor at 1 bar + dT_1…dT_7 (PCHIP monotonic spline, no Gaussian smoothing).  
T_anchor prior: U(1000, 3000) K (widened from Guidebook default [1500, 3000]).  
**pRT pressure grid: 10⁻³–100 bar** (narrowed from 10⁻⁵ in 002PM; matches Piette+2020 original).  
Upper ΔT priors dT_5/6/7: U(0, 1000) K (widened from [0, 500–600] in 002PM).  

**Source**: `Guidebook_GAStronomy_Piette_v1.0.py` | N_live=700, ev_tol=0.5, norm=savgol, per_chip_scale=False

In [ ]:
import sys
sys.path.insert(0, '/data2/peng/Recipe_DH_Tau_B')  # ensure Recipe_DH_Tau_B tasting_analysis is used

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from analysis import (
    plot_ccf, plot_equil_chemistry,
    plot_binned_model_vs_data,
)
from tasting_analysis import run_ccf_workflow, run_equil_chemistry_workflow
from scipy.interpolate import PchipInterpolator

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------
workpath     = Path('/data2/peng')
retrieval_id = '936677_N700_ev0.5_Normsavgol_PerChipScaleFalse'
night        = '2022-12-31'

print('workpath:', workpath)
print('retrieval_id:', retrieval_id)
print('night:', night)
print('retrieval exists:', (workpath / 'retrievals' / retrieval_id).exists())
# ------------------------------------------------------------------
# 1) CCF robustness workflow
# ------------------------------------------------------------------
ccf_out = run_ccf_workflow(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night=night,
    rvlag=np.arange(-10, 10, 0.1),
    clean_grids=((-10, -1), (0, 10)),
    n_shuffle=200,
    random_seed=42,
    combined_spectrum_name="extracted_spectra_combined_sigmaclipper.npy",
)

print('Combined spectra file:', ccf_out['combined_path'])
print('Valid pixels used:', ccf_out['valid_pixels'])
print(f"Peak SNR = {ccf_out['peak_snr']:.2f} at RV = {ccf_out['peak_rv']:.1f} km/s")
print(f"Null peak SNR mean\u00b1std = {ccf_out['null_mu']:.2f} \u00b1 {ccf_out['null_sigma']:.2f}")
print(f"Detection z-score (vs shuffled null) = {ccf_out['z_score']:.2f}")

plot_ccf(ccf_out, retrieval_id)
# ------------------------------------------------------------------
# 2) Equilibrium-chemistry posterior diagnostics
# ------------------------------------------------------------------
chem_out = run_equil_chemistry_workflow(workpath=workpath, retrieval_id=retrieval_id)

print('Posterior shape:', chem_out['posterior'].shape)
print('Retrieved params:', chem_out['columns'])
display(chem_out['summary'])

print('\nText summary:')
for _, row in chem_out['summary'].iterrows():
    print(
        f"{row['parameter']}: median={row['p50']:.4g}, "
        f"1\u03c3=[-{row['minus_1sigma']:.3g}, +{row['plus_1sigma']:.3g}], "
        f"3\u03c3=[-{row['minus_3sigma']:.3g}, +{row['plus_3sigma']:.3g}]"
    )

plot_equil_chemistry(chem_out, retrieval_id)


In [ ]:
# ------------------------------------------------------------------
# Night 1 (2022-12-31): binned data vs model, per-order-detector
# ------------------------------------------------------------------
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)


In [ ]:
# ------------------------------------------------------------------
# Night 2 (2023-01-01): model_flux_N2 vs data — binned, per-order-detector
# ------------------------------------------------------------------
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2023-01-01',
    model_wave_npy='retrieval_model_wave_N2.npy',
    model_flux_npy='retrieval_model_flux_N2.npy',
)


In [ ]:
# ------------------------------------------------------------------
# P-T profile posterior — 1σ / 2σ / 3σ bands
# Piette+2020 parameterisation (T_anchor + dT_1…dT_7, PCHIP spline)
# Pressure grid: 10⁻³–100 bar (003PM update; matches Piette+2020 original).
# ------------------------------------------------------------------

_PIETTE_EQUIL_CHEM_PARAMS = [
    "rv_N1", "rv_N2", "vsini", "epsilon", "log_g",
    "T_anchor", "dT_1", "dT_2", "dT_3", "dT_4", "dT_5", "dT_6", "dT_7",
    "C_H", "C/O", "log_12CO_13CO",
]

LOG_P_NODES = np.array([2.0, 1.5, 1.0, 0.5, 0.0, -1.0, -2.0, -3.0])
ANCHOR_IDX  = 4   # log P = 0.0 (1 bar)
# 003PM: grid matched to PCHIP node range (10⁻³–100 bar)
pressure    = np.logspace(-3, 2, 50)

retrieval_dir = workpath / 'retrievals' / retrieval_id
posterior = np.load(retrieval_dir / 'final_posterior.npy')
col = {k: i for i, k in enumerate(_PIETTE_EQUIL_CHEM_PARAMS)}

n_samples = posterior.shape[0]
all_temps = np.empty((n_samples, len(pressure)))

for i, s in enumerate(posterior):
    T_anchor = s[col['T_anchor']]
    dT = np.array([s[col[f'dT_{j+1}']] for j in range(7)])

    T_nodes = np.empty(8)
    T_nodes[ANCHOR_IDX] = T_anchor
    for j in range(ANCHOR_IDX - 1, -1, -1):
        T_nodes[j] = T_nodes[j + 1] + dT[j]
    for j in range(ANCHOR_IDX + 1, 8):
        T_nodes[j] = T_nodes[j - 1] - dT[j - 1]

    log_P_asc = LOG_P_NODES[::-1]
    T_asc     = T_nodes[::-1]
    pchip     = PchipInterpolator(log_P_asc, T_asc)
    log_P_atm = np.log10(pressure)
    temp      = pchip(log_P_atm)
    # Safety clamp (no-op for 10⁻³–100 bar grid; kept for robustness)
    temp      = np.where(log_P_atm < log_P_asc[0],  T_asc[0],  temp)
    temp      = np.where(log_P_atm > log_P_asc[-1], T_asc[-1], temp)
    all_temps[i] = np.clip(temp, 1.0, 30000.0)

pcts = np.percentile(all_temps, [0.27, 2.28, 15.87, 50.0, 84.13, 97.72, 99.73], axis=0)

fig, ax = plt.subplots(figsize=(5, 7))
color = 'steelblue'
ax.fill_betweenx(pressure, pcts[0], pcts[6], color=color, alpha=0.15,
                 linewidth=0, label='3σ (99.7%)')
ax.fill_betweenx(pressure, pcts[1], pcts[5], color=color, alpha=0.30,
                 linewidth=0, label='2σ (95.4%)')
ax.fill_betweenx(pressure, pcts[2], pcts[4], color=color, alpha=0.55,
                 linewidth=0, label='1σ (68.3%)')
ax.plot(pcts[3], pressure, color='navy', lw=1.8, label='Median')

# Mark the 8 fixed P-T anchor nodes
for logp in LOG_P_NODES:
    ax.axhline(10**logp, color='gray', lw=0.5, ls=':', alpha=0.6)

ax.set_ylim(1e-3, 100)
ax.set_xlim(500, 4000)
ax.set_yscale('log')
ax.invert_yaxis()
ax.set_xlabel('Temperature (K)', fontsize=11)
ax.set_ylabel('Pressure (bar)', fontsize=11)
ax.set_title(
    f'P-T profile posterior (003PM-EQ, Piette+2020 PCHIP)\n{retrieval_id}',
    fontsize=9,
)
ax.legend(fontsize=9, loc='upper right')
ax.grid(alpha=0.2, lw=0.5)
ax.tick_params(labelsize=9)
plt.tight_layout()
plt.show()

print(f'Posterior samples used: {n_samples}')
print(f'Median T range: {pcts[3].min():.0f} – {pcts[3].max():.0f} K')
print(f'T_anchor median: {np.percentile(posterior[:, col["T_anchor"]], 50):.1f} K')
print(f'C_H median:      {np.percentile(posterior[:, col["C_H"]], 50):.3f} dex')
print(f'C/O median:      {np.percentile(posterior[:, col["C/O"]], 50):.3f}')
print(f'log_12CO_13CO median: {np.percentile(posterior[:, col["log_12CO_13CO"]], 50):.3f}')


In [ ]:
# ------------------------------------------------------------------
# Combined two-night spectrum vs best-fit model (Night 1 model)
# Uses same plot_binned_model_vs_data layout as Night 1 / Night 2 above
# (telluric overlay, residual panel, savgol smoothing, per-order layout).
# Wavelength grid and telluric from Night 1 (2022-12-31).
# ------------------------------------------------------------------
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
    data_npy='extracted_spectra_combined_two_nights.npy',
    err_npy='extracted_spectra_combined_two_nights_err.npy',
    data_dir=workpath / 'combined_two_nights',
)


## Retrieval: 1461589_N700_ev0.5_Normsavgol_PerChipScaleFalse

**Series 001DG-EQ** — Dynamic Gradients (DG) P-T Profile (Picos+2025) + Equilibrium chemistry  
Equilibrium chemistry (C_H, C/O, log_12CO_13CO), two-night fit.  
P-T: 6-node piecewise-gradient structure anchored at the RCE (nabla_RCE, nabla_0–4, T_bottom, log_P_RCE, dlog_P_bot, dlog_P_top).  
Convective-zone gradients constrained ≥ 0.04 and ≤ nabla_RCE; radiative-zone gradients ≥ 0.00 and ≤ nabla_RCE.  
pRT pressure grid: 10⁻⁵–100 bar (50 layers).  

**Source**: `Guidebook_GAStronomy_Picos_DG_v1.0.py` | N_live=700, ev_tol=0.5, norm=savgol, per_chip_scale=False

In [ ]:
import sys
sys.path.insert(0, '/data2/peng/Recipe_DH_Tau_B')  # ensure Recipe_DH_Tau_B analysis is used

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from analysis import (
    plot_ccf, plot_equil_chemistry,
    plot_binned_model_vs_data,
    _EQUIL_CHEM_PARAMS,
)
from tasting_analysis import run_ccf_workflow, run_equil_chemistry_workflow

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------
workpath     = Path('/data2/peng')
retrieval_id = '1461589_N700_ev0.5_Normsavgol_PerChipScaleFalse'
night        = '2022-12-31'

print('workpath:', workpath)
print('retrieval_id:', retrieval_id)
print('night:', night)
print('retrieval exists:', (workpath / 'retrievals' / retrieval_id).exists())
# ------------------------------------------------------------------
# 1) CCF robustness workflow
# ------------------------------------------------------------------
ccf_out = run_ccf_workflow(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night=night,
    rvlag=np.arange(-10, 10, 0.1),
    clean_grids=((-10, -1), (0, 10)),
    n_shuffle=200,
    random_seed=42,
    combined_spectrum_name='extracted_spectra_combined_sigmaclipper.npy',
)

print('Combined spectra file:', ccf_out['combined_path'])
print('Valid pixels used:', ccf_out['valid_pixels'])
print(f"Peak SNR = {ccf_out['peak_snr']:.2f} at RV = {ccf_out['peak_rv']:.1f} km/s")
print(f"Null peak SNR mean±std = {ccf_out['null_mu']:.2f} ± {ccf_out['null_sigma']:.2f}")
print(f"Detection z-score (vs shuffled null) = {ccf_out['z_score']:.2f}")

plot_ccf(ccf_out, retrieval_id)
# ------------------------------------------------------------------
# 2) Equilibrium-chemistry posterior diagnostics
# ------------------------------------------------------------------
chem_out = run_equil_chemistry_workflow(workpath=workpath, retrieval_id=retrieval_id)

print('Posterior shape:', chem_out['posterior'].shape)
print('Retrieved params:', chem_out['columns'])
display(chem_out['summary'])

print('\nText summary:')
for _, row in chem_out['summary'].iterrows():
    print(
        f"{row['parameter']}: median={row['p50']:.4g}, "
        f"1σ=[-{row['minus_1sigma']:.3g}, +{row['plus_1sigma']:.3g}], "
        f"3σ=[-{row['minus_3sigma']:.3g}, +{row['plus_3sigma']:.3g}]"
    )

plot_equil_chemistry(chem_out, retrieval_id)

In [ ]:
# ------------------------------------------------------------------
# Night 1 (2022-12-31): binned data vs model, per-order-detector
# ------------------------------------------------------------------
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)

In [ ]:
# ------------------------------------------------------------------
# Night 2 (2023-01-01): model_flux_N2 vs data — binned, per-order-detector
# ------------------------------------------------------------------
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2023-01-01',
    model_wave_npy='retrieval_model_wave_N2.npy',
    model_flux_npy='retrieval_model_flux_N2.npy',
)

In [ ]:
# ------------------------------------------------------------------
# P-T profile posterior — 1σ / 2σ / 3σ bands
# DG parameterisation (nabla_* + T_bottom, Picos+2025)
# Pressure grid: 10⁻⁵–100 bar (50 layers)
# ------------------------------------------------------------------
from analysis import plot_pt_profile

plot_pt_profile(
    workpath=workpath,
    retrieval_id=retrieval_id,
    param_keys=_EQUIL_CHEM_PARAMS,
)

In [ ]:
# ------------------------------------------------------------------
# Combined two-night spectrum vs best-fit model (Night 1 model)
# Uses same plot_binned_model_vs_data layout as Night 1 / Night 2 above
# (telluric overlay, residual panel, savgol smoothing, per-order layout).
# Wavelength grid and telluric from Night 1 (2022-12-31).
# ------------------------------------------------------------------
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
    data_npy='extracted_spectra_combined_two_nights.npy',
    err_npy='extracted_spectra_combined_two_nights_err.npy',
    data_dir=workpath / 'combined_two_nights',
)


## Retrieval: 2585252_N200_ev0.5_Normmedian_PerChipScaleFalse

**Series 001DG-EQ** — Dynamic Gradients (DG) P-T Profile (Picos+2025) + Equilibrium chemistry  
Equilibrium chemistry (C_H, C/O, log_12CO_13CO), two-night fit.  
P-T: 6-node piecewise-gradient structure anchored at the RCE (nabla_RCE, nabla_0–5, T_bottom, log_P_RCE, dlog_P_bot, dlog_P_top).  
Convective-zone gradients constrained ≥ 0.04 and ≤ nabla_RCE; radiative-zone gradients ≥ 0.00 and ≤ nabla_RCE.  
pRT pressure grid: 10⁻⁵–100 bar (50 layers).  

**Source**: `Guidebook_GAStronomy_Picos_DG_v1.0.py` | N_live=200, ev_tol=0.5, norm=median, per_chip_scale=False

In [ ]:
import sys
sys.path.insert(0, '/data2/peng/Recipe_DH_Tau_B')  # ensure Recipe_DH_Tau_B analysis is used

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from analysis import (
    plot_ccf, plot_equil_chemistry,
    plot_binned_model_vs_data,
    _EQUIL_CHEM_PARAMS,
)
from tasting_analysis import run_ccf_workflow, run_equil_chemistry_workflow

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------
workpath     = Path('/data2/peng')
retrieval_id = '2585252_N200_ev0.5_Normmedian_PerChipScaleFalse'
night        = '2022-12-31'

print('workpath:', workpath)
print('retrieval_id:', retrieval_id)
print('night:', night)
print('retrieval exists:', (workpath / 'retrievals' / retrieval_id).exists())
# ------------------------------------------------------------------
# 1) CCF robustness workflow
# ------------------------------------------------------------------
ccf_out = run_ccf_workflow(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night=night,
    rvlag=np.arange(-10, 10, 0.1),
    clean_grids=((-10, -1), (0, 10)),
    n_shuffle=200,
    random_seed=42,
    combined_spectrum_name='extracted_spectra_combined_flux_cal.npy',
)

print('Combined spectra file:', ccf_out['combined_path'])
print('Valid pixels used:', ccf_out['valid_pixels'])
print(f"Peak SNR = {ccf_out['peak_snr']:.2f} at RV = {ccf_out['peak_rv']:.1f} km/s")
print(f"Null peak SNR mean±std = {ccf_out['null_mu']:.2f} ± {ccf_out['null_sigma']:.2f}")
print(f"Detection z-score (vs shuffled null) = {ccf_out['z_score']:.2f}")

plot_ccf(ccf_out, retrieval_id)
# ------------------------------------------------------------------
# 2) Equilibrium-chemistry posterior diagnostics
# ------------------------------------------------------------------
chem_out = run_equil_chemistry_workflow(workpath=workpath, retrieval_id=retrieval_id)

print('Posterior shape:', chem_out['posterior'].shape)
print('Retrieved params:', chem_out['columns'])
display(chem_out['summary'])

print('\nText summary:')
for _, row in chem_out['summary'].iterrows():
    print(
        f"{row['parameter']}: median={row['p50']:.4g}, "
        f"1\u03c3=[-{row['minus_1sigma']:.3g}, +{row['plus_1sigma']:.3g}], "
        f"3\u03c3=[-{row['minus_3sigma']:.3g}, +{row['plus_3sigma']:.3g}]"
    )

plot_equil_chemistry(chem_out, retrieval_id)

In [ ]:
# ------------------------------------------------------------------
# Night 1 (2022-12-31): binned data vs model, per-order-detector
# ------------------------------------------------------------------
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)

In [ ]:
# ------------------------------------------------------------------
# Night 2 (2023-01-01): model_flux_N2 vs data — binned, per-order-detector
# ------------------------------------------------------------------
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2023-01-01',
    model_wave_npy='retrieval_model_wave_N2.npy',
    model_flux_npy='retrieval_model_flux_N2.npy',
)

In [ ]:
# ------------------------------------------------------------------
# P-T profile posterior — 1σ / 2σ / 3σ bands
# DG parameterisation (nabla_* + T_bottom, Picos+2025)
# Pressure grid: 10⁻⁵–100 bar (50 layers)
# ------------------------------------------------------------------
from analysis import plot_pt_profile

plot_pt_profile(
    workpath=workpath,
    retrieval_id=retrieval_id,
    param_keys=_EQUIL_CHEM_PARAMS,
)

In [ ]:
# ------------------------------------------------------------------
# Combined two-night spectrum vs best-fit model (Night 1 model)
# Uses same plot_binned_model_vs_data layout as Night 1 / Night 2 above
# (telluric overlay, residual panel, savgol smoothing, per-order layout).
# Wavelength grid and telluric from Night 1 (2022-12-31).
# ------------------------------------------------------------------
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
    data_npy='extracted_spectra_combined_two_nights.npy',
    err_npy='extracted_spectra_combined_two_nights_err.npy',
    data_dir=workpath / 'combined_two_nights',
)


## Retrieval: 2644115_N700_ev0.5_Normmedian_PerChipScaleFalse

**Series 001DG-EQ** — Dynamic Gradients (DG) P-T Profile (Picos+2025) + Equilibrium chemistry  
Equilibrium chemistry (C_H, C/O, log_12CO_13CO), two-night fit.  
P-T: 6-node piecewise-gradient structure anchored at the RCE (nabla_RCE, nabla_0–5, T_bottom, log_P_RCE, dlog_P_bot, dlog_P_top).  
Convective-zone gradients constrained ≥ 0.04 and ≤ nabla_RCE; radiative-zone gradients ≥ 0.00 and ≤ nabla_RCE.  
pRT pressure grid: 10⁻⁵–100 bar (50 layers).  

**Source**: `Guidebook_GAStronomy_Picos_DG_v1.0.py` | N_live=700, ev_tol=0.5, norm=median, per_chip_scale=False  
**Follow-up** of 2585252 (N=200) — higher live-point count for better posterior sampling.

In [ ]:
import sys
sys.path.insert(0, '/data2/peng/Recipe_DH_Tau_B')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from analysis import (
    plot_ccf, plot_equil_chemistry,
    plot_binned_model_vs_data,
    _EQUIL_CHEM_PARAMS,
)
from tasting_analysis import run_ccf_workflow, run_equil_chemistry_workflow

workpath     = Path('/data2/peng')
retrieval_id = '2644115_N700_ev0.5_Normmedian_PerChipScaleFalse'
night        = '2022-12-31'

print('retrieval exists:', (workpath / 'retrievals' / retrieval_id).exists())

ccf_out = run_ccf_workflow(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night=night,
    rvlag=np.arange(-10, 10, 0.1),
    clean_grids=((-10, -1), (0, 10)),
    n_shuffle=200,
    random_seed=42,
    combined_spectrum_name='extracted_spectra_combined_sigmaclipper.npy',
)
print(f"Peak SNR = {ccf_out['peak_snr']:.2f} at RV = {ccf_out['peak_rv']:.1f} km/s")
print(f"z-score = {ccf_out['z_score']:.2f}")
plot_ccf(ccf_out, retrieval_id)

chem_out = run_equil_chemistry_workflow(workpath=workpath, retrieval_id=retrieval_id)
print('Posterior shape:', chem_out['posterior'].shape)
display(chem_out['summary'])
print('\nText summary:')
for _, row in chem_out['summary'].iterrows():
    print(
        f"{row['parameter']}: median={row['p50']:.4g}, "
        f"1σ=[-{row['minus_1sigma']:.3g}, +{row['plus_1sigma']:.3g}]"
    )
plot_equil_chemistry(chem_out, retrieval_id)

In [ ]:
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)

In [ ]:
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2023-01-01',
    model_wave_npy='retrieval_model_wave_N2.npy',
    model_flux_npy='retrieval_model_flux_N2.npy',
)

In [ ]:
from analysis import plot_pt_profile
plot_pt_profile(
    workpath=workpath,
    retrieval_id=retrieval_id,
    param_keys=_EQUIL_CHEM_PARAMS,
)

In [ ]:
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
    data_npy='extracted_spectra_combined_two_nights.npy',
    err_npy='extracted_spectra_combined_two_nights_err.npy',
    data_dir=workpath / 'combined_two_nights',
)


### Analysis: 2644115 vs 2585252 — Problems Identified

#### Summary comparison (MAP values)

| Parameter | 2585252 (N=200) | 2644115 (N=700) | Comment |
|---|---|---|---|
| lnZ | −17 051.9 | −17 048.2 | Δ=3.7 nats — consistent within MultiNest sampling noise |
| C/O | 0.554 | 0.580 ± 0.007 | Near-solar; tightly constrained in N=700 |
| [C/H] | +0.13 | **+0.29 ± 0.04** | Discrepancy due to N=200 under-sampling |
| log ¹²CO/¹³CO | 1.81 | 1.98 ± 0.07 | ~95, above solar (70) |
| log_g | 4.35 | **4.39 ± 0.07** | **3.4σ above prior N(3.7, 0.2)** — see Problem 1 |
| T_bottom | 3396 K | 3299 ± 65 K | Consistent |
| vsini | 7.34 km/s | 7.42 ± 0.15 km/s | Consistent |
| χ² (N1/N2) | 1.57 / 1.63 | 1.57 / 1.62 | Identical fit quality |
| Posterior samples | 2 367 | **15 056** | N=700 much better sampled |

#### Problem 1 — log_g = 4.39 pulls 3.4σ from Gaussian prior (prior tension)

The retrieved log_g = 4.39 ± 0.07 is 3.4σ above the prior center N(3.7, 0.2). For DH Tau B (~14 Myr, Taurus association, estimated mass ~11–15 M_Jup), evolution models (e.g. SONORA Diamondback, Saumon & Marley 2008) predict log_g ≈ 3.5–3.7. A value of 4.39 corresponds to a much more compact object (~Saturn-like radius at Jupiter mass), inconsistent with a young inflated companion.

**Possible causes**:
- `log_g`–`[C/H]` degeneracy: higher gravity requires higher molecular abundance to produce the same line depth → `[C/H]` is also inflated (+0.29 vs expected ~0.0 to +0.1).
- The prior sigma is too narrow (0.2 dex) relative to the true uncertainty — the data prefers a different region of parameter space and the prior is insufficient to hold it.
- Residual systematic in the data driving the fit toward an unphysical solution.

**Proposed fix**: widen the log_g prior to N(3.7, 0.5) or use a uniform prior U(3.0, 5.0) and check whether log_g converges to a physically meaningful value with no prior pulling.

#### Problem 2 — nabla_5 piles against lower bound (unconstrained top atmosphere)

nabla_5 (topmost gradient, ~10⁻⁵ bar) has p50 = 0.010, with 51% of samples below 0.01 (lower bound = 0). K-band flux is insensitive to pressures below ~10⁻² bar, so the top-atmosphere gradient is unconstrained by the data. The posterior is set entirely by the prior lower bound.

**Impact**: nabla_5 is unphysically small (near-zero gradient = isothermal top atmosphere), but this has negligible effect on the K-band model spectrum. It does waste one degree of freedom.

**Proposed fix**: fix nabla_5 to the radiative equilibrium value (e.g. 0.25, matching nabla_3/4 which are better constrained) or remove it and use nabla_4 as the top node. Alternatively, narrow the prior to [0.05, 0.34] to prevent the boundary-piling pathology.

#### Problem 3 — [C/H] discrepancy between N=200 and N=700 runs

N=200 returned [C/H] = +0.13 while N=700 returns [C/H] = +0.29 ± 0.04. The 2-sigma difference confirms that the N=200 posterior was insufficiently sampled for the 19-dimensional DG parameter space. **N=700 results should be treated as the primary reference.**


## Retrieval: 2734585_N300_ev0.5_Normmedian_PerChipScaleFalse

**Series 001DG-EQ** — Dynamic Gradients (DG) P-T Profile (Picos+2025) + Equilibrium chemistry  
Equilibrium chemistry (C_H, C/O, log_12CO_13CO), two-night fit.  
P-T: 6-node piecewise-gradient structure anchored at the RCE (nabla_RCE, nabla_0–5, T_bottom, log_P_RCE, dlog_P_bot, dlog_P_top).  

**Source**: `Guidebook_GAStronomy_Picos_DG_v1.0.py` | N_live=300, ev_tol=0.5, norm=median, per_chip_scale=False  
**Data**: `extracted_spectra_combined_sigmaclipper.npy` (no SINFONI flux calibration).  
**Note**: lnZ = −14 854 vs −17 051 for 2644115 (Δ = +2197 nats) — consistent with blaze-corrected input spectra.


In [ ]:
import sys
sys.path.insert(0, '/data2/peng/Recipe_DH_Tau_B')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from analysis import (
    plot_ccf, plot_equil_chemistry,
    plot_binned_model_vs_data,
    _EQUIL_CHEM_PARAMS,
)
from tasting_analysis import run_ccf_workflow, run_equil_chemistry_workflow

workpath     = Path('/data2/peng')
retrieval_id = '2734585_N300_ev0.5_Normmedian_PerChipScaleFalse'
night        = '2022-12-31'

print('retrieval exists:', (workpath / 'retrievals' / retrieval_id).exists())

ccf_out = run_ccf_workflow(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night=night,
    rvlag=np.arange(-10, 10, 0.1),
    clean_grids=((-10, -1), (0, 10)),
    n_shuffle=200,
    random_seed=42,
    combined_spectrum_name='extracted_spectra_combined_sigmaclipper.npy',
)
print(f"Peak SNR = {ccf_out['peak_snr']:.2f} at RV = {ccf_out['peak_rv']:.1f} km/s")
print(f"z-score = {ccf_out['z_score']:.2f}")
plot_ccf(ccf_out, retrieval_id)

chem_out = run_equil_chemistry_workflow(workpath=workpath, retrieval_id=retrieval_id)
print('Posterior shape:', chem_out['posterior'].shape)
display(chem_out['summary'])
print('\nText summary:')
for _, row in chem_out['summary'].iterrows():
    print(
        f"{row['parameter']}: median={row['p50']:.4g}, "
        f"1σ=[-{row['minus_1sigma']:.3g}, +{row['plus_1sigma']:.3g}]"
    )
plot_equil_chemistry(chem_out, retrieval_id)


In [ ]:
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)


In [ ]:
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2023-01-01',
    model_wave_npy='retrieval_model_wave_N2.npy',
    model_flux_npy='retrieval_model_flux_N2.npy',
)


In [ ]:
from analysis import plot_pt_profile
plot_pt_profile(
    workpath=workpath,
    retrieval_id=retrieval_id,
    param_keys=_EQUIL_CHEM_PARAMS,
)


In [ ]:
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
    data_npy='extracted_spectra_combined_two_nights.npy',
    err_npy='extracted_spectra_combined_two_nights_err.npy',
    data_dir=workpath / 'combined_two_nights',
)


### Analysis: 2734585 vs 2644115 — Effect of blaze correction

#### Summary comparison (MAP values)

| Parameter | 2644115 (N=700, no blaze corr.) | 2734585 (N=300, blaze corr.) | Comment |
|---|---|---|---|
| lnZ | −17 051.9 | **−14 854.1** | **Δ = +2197 nats — decisive improvement** |
| C/O | 0.580 ± 0.007 | 0.472 ± 0.009 | Sub-solar; substantial shift |
| [C/H] | +0.29 ± 0.04 | **−0.17 ± 0.04** | Sign reversal — blaze was inflating abundance |
| log ¹²CO/¹³CO | 1.98 ± 0.07 | 1.00 ± 0.04 | ~10 vs ~95 — major change |
| log_g | 4.39 ± 0.07 | 4.65 ± 0.06 | Still above prior centre (3.7); tension persists |
| T_bottom | 3299 ± 65 K | 3869 ± 138 K | ~570 K hotter |
| vsini | 7.42 ± 0.15 km/s | 6.82 ± 0.14 km/s | Consistent |
| ε (limb-darkening) | 0.82 ± 0.12 | 0.747 ± 0.11 | Consistent |
| χ² (N1/N2) | 1.57 / 1.62 | **1.47 / 1.36** | Better per-pixel fit |
| Posterior samples | 15 056 | 4 112 | N=300 under-sampled for 19-D space |

#### Problem 1 — lnZ jump of +2197 nats: blaze correction changes everything

The +2197 nat increase in lnZ is decisive evidence that the blaze-corrected data is much more consistent
with the model. The blaze was introducing a systematic low-frequency slope (peak/min up to 2× at det=2)
that the retrieval was compensating by adjusting the temperature profile and molecular abundances.

Key changes driven by blaze correction:
- **[C/H] sign reversal**: +0.29 → −0.17. The blaze-inflated slope was causing the retrieval to prefer
  super-solar metallicity to match the artificial curvature. After correction, the chemistry is sub-solar.
- **T_bottom increase**: 3299 → 3869 K. The blaze envelope was suppressing the apparent continuum level,
  causing the retrieval to prefer a cooler atmosphere.
- **log ¹²CO/¹³CO**: 1.98 → 1.00 (95 → 10). The isotopologue ratio is highly sensitive to the
  spectral shape; removal of the blaze systematically shifts this.

#### Problem 2 — log_g tension persists (now worse: 4.65 vs 4.39)

Despite the greatly improved fit, log_g = 4.65 ± 0.06 is now 4.1σ above prior centre N(3.7, 0.2).
The blaze correction did not resolve the log_g–[C/H] degeneracy; it shifted both parameters to a new
unphysical solution. **Proposed fix: widen log_g prior to U(3.0, 5.0) or N(3.7, 0.5).**

#### Problem 3 — N=300 under-sampled for 19-D space (4112 samples)

4112 posterior samples is insufficient for a 19-D parameter space (rule of thumb: ≥ N_live × N_dim = 5700).
The chemistry shift [C/H] = −0.17 could shift further with N=700. **Recommended: re-run at N=700.**

#### Next step

Run 001DG-EQ with blaze-corrected data at N=700 and widened log_g prior (U[3.0, 5.0] or N(3.7, 0.5))
to resolve the log_g tension and obtain well-sampled posteriors for the blaze-corrected dataset.


## Retrieval: 2743787_N300_ev0.5_Normmedian_PerChipScaleFalse

**Series 001DG-EQ** — Dynamic Gradients (DG) P-T Profile (Picos+2025) + Equilibrium chemistry  
Equilibrium chemistry (C_H, C/O, log_12CO_13CO), two-night fit.  
P-T: 6-node piecewise-gradient structure (nabla_RCE, nabla_0–5, T_bottom, log_P_RCE, dlog_P_bot, dlog_P_top).  

**Source**: `Guidebook_GAStronomy_Picos_DG_v1.0.py` | N_live=300, ev_tol=0.5, norm=median, per_chip_scale=False  
**Data**: sigmaclipper blaze-corrected (`extracted_spectra_combined_sigmaclipper.npy`).  
**Note**: Sibling run to 2734585 — different MultiNest seed converged to a different posterior mode.  
lnZ = −14 868 vs −14 854 for 2734585 (Δ = −14 nats; 2734585 weakly preferred).  
log_g = 3.94 here vs 4.65 in 2734585 — evidence of a bimodal log_g posterior at N=300.


In [ ]:
import sys
sys.path.insert(0, '/data2/peng/Recipe_DH_Tau_B')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from analysis import (
    plot_ccf, plot_equil_chemistry,
    plot_binned_model_vs_data,
    _EQUIL_CHEM_PARAMS,
)
from tasting_analysis import run_ccf_workflow, run_equil_chemistry_workflow

workpath     = Path('/data2/peng')
retrieval_id = '2743787_N300_ev0.5_Normmedian_PerChipScaleFalse'
night        = '2022-12-31'

print('retrieval exists:', (workpath / 'retrievals' / retrieval_id).exists())

ccf_out = run_ccf_workflow(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night=night,
    rvlag=np.arange(-10, 10, 0.1),
    clean_grids=((-10, -1), (0, 10)),
    n_shuffle=200,
    random_seed=42,
    combined_spectrum_name='extracted_spectra_combined_sigmaclipper.npy',
)
print(f"Peak SNR = {ccf_out['peak_snr']:.2f} at RV = {ccf_out['peak_rv']:.1f} km/s")
print(f"z-score = {ccf_out['z_score']:.2f}")
plot_ccf(ccf_out, retrieval_id)

chem_out = run_equil_chemistry_workflow(workpath=workpath, retrieval_id=retrieval_id)
print('Posterior shape:', chem_out['posterior'].shape)
display(chem_out['summary'])
print('\nText summary:')
for _, row in chem_out['summary'].iterrows():
    print(
        f"{row['parameter']}: median={row['p50']:.4g}, "
        f"1σ=[-{row['minus_1sigma']:.3g}, +{row['plus_1sigma']:.3g}]"
    )
plot_equil_chemistry(chem_out, retrieval_id)


In [ ]:
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)


In [ ]:
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2023-01-01',
    model_wave_npy='retrieval_model_wave_N2.npy',
    model_flux_npy='retrieval_model_flux_N2.npy',
)


In [ ]:
from analysis import plot_pt_profile
plot_pt_profile(
    workpath=workpath,
    retrieval_id=retrieval_id,
    param_keys=_EQUIL_CHEM_PARAMS,
)


In [ ]:
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
    data_npy='extracted_spectra_combined_two_nights.npy',
    err_npy='extracted_spectra_combined_two_nights_err.npy',
    data_dir=workpath / 'combined_two_nights',
)


### Analysis: 2743787 vs 2734585 — Bimodal log_g in DG model (N=300)

#### Summary comparison (MAP values)

| Parameter | 2734585 (DG-EQ, blaze-corr.) | 2743787 (DG-EQ, blaze-corr.) | Comment |
|---|---|---|---|
| lnZ | −14 854.1 | −14 868.3 | Δ = −14 nats; 2734585 weakly preferred |
| C/O | 0.472 ± 0.009 | 0.514 ± 0.010 | Sub-solar in both |
| [C/H] | −0.17 ± 0.04 | **−0.44 ± 0.044** | Substantially more sub-solar |
| log ¹²CO/¹³CO | 1.00 ± 0.04 | 1.03 ± 0.042 | Consistent |
| log_g | 4.652 ± 0.064 | **3.943 ± 0.029** | **Opposite extremes of prior** |
| T_bottom | 3 869 ± 138 K | **4 733 ± 131 K** | 860 K hotter |
| vsini | 6.82 ± 0.14 km/s | 7.05 ± 0.22 km/s | Consistent |
| χ² (N1/N2) | 1.47 / 1.36 | 1.47 / 1.36 | Identical fit quality |
| Posterior samples | 4 112 | 3 348 | Both under-sampled |

#### Interpretation: bimodal posterior in log_g–[C/H]–T_bottom

Two runs with the same model (001DG-EQ) and same data converged to very different solutions:
- **2734585**: log_g = 4.65, [C/H] = −0.17, T_bottom = 3 869 K (high-gravity, warm, mild sub-solar)
- **2743787**: log_g = 3.94, [C/H] = −0.44, T_bottom = 4 733 K (physical gravity, hot, very sub-solar)

Both have essentially identical lnZ (Δ = 14 nats) and identical χ², indicating two nearly
degenerate modes. The log_g–[C/H]–T degeneracy drives this: at fixed spectral quality, higher
gravity → deeper lines → lower metallicity needed → hotter atmosphere to match the continuum.

Neither N=300 run fully characterises this bimodal structure. **N=700 is required to map
both modes and determine which is the global maximum.** Alternatively, a flat log_g prior
U(3.0, 5.0) may break the degeneracy.


## Retrieval: 2768012_N300_ev0.5_Normmedian_PerChipScaleFalse

**Series 003PM-EQ** — Piette & Madhusudhan (2020) / Xuan et al. (2024) P-T Profile + Equilibrium chemistry  
P-T: T_anchor (1 bar) + 7 monotonic ΔT increments over 8 fixed pressure nodes (100–0.001 bar),  
     PCHIP monotonic cubic spline interpolation.  
Equilibrium chemistry (C_H, C/O, log_12CO_13CO), two-night fit.  
16 free parameters total (vs 19 for DG P-T).  

**Source**: `Guidebook_GAStronomy_Piette_v1.0.py` | N_live=300, ev_tol=0.5, norm=median, per_chip_scale=False  
**Data**: sigmaclipper blaze-corrected (`extracted_spectra_combined_sigmaclipper.npy`).  
**lnZ = −14 842.7** — best of all runs so far (Δ = +11 vs 2734585, Δ = +25 vs 2743787).


In [ ]:
import sys
sys.path.insert(0, '/data2/peng/Recipe_DH_Tau_B')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from analysis import (
    plot_ccf, plot_equil_chemistry,
    plot_binned_model_vs_data,
    _PIETTE_EQUIL_PARAMS,
    plot_pt_profile_piette,
)
from tasting_analysis import run_ccf_workflow, run_equil_chemistry_workflow

workpath     = Path('/data2/peng')
retrieval_id = '2768012_N300_ev0.5_Normmedian_PerChipScaleFalse'
night        = '2022-12-31'

print('retrieval exists:', (workpath / 'retrievals' / retrieval_id).exists())

ccf_out = run_ccf_workflow(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night=night,
    rvlag=np.arange(-10, 10, 0.1),
    clean_grids=((-10, -1), (0, 10)),
    n_shuffle=200,
    random_seed=42,
    combined_spectrum_name='extracted_spectra_combined_sigmaclipper.npy',
)
print(f"Peak SNR = {ccf_out['peak_snr']:.2f} at RV = {ccf_out['peak_rv']:.1f} km/s")
print(f"z-score = {ccf_out['z_score']:.2f}")
plot_ccf(ccf_out, retrieval_id)

chem_out = run_equil_chemistry_workflow(workpath=workpath, retrieval_id=retrieval_id)
print('Posterior shape:', chem_out['posterior'].shape)
display(chem_out['summary'])
print('\nText summary:')
for _, row in chem_out['summary'].iterrows():
    print(
        f"{row['parameter']}: median={row['p50']:.4g}, "
        f"1σ=[-{row['minus_1sigma']:.3g}, +{row['plus_1sigma']:.3g}]"
    )
plot_equil_chemistry(chem_out, retrieval_id)


In [ ]:
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)


In [ ]:
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2023-01-01',
    model_wave_npy='retrieval_model_wave_N2.npy',
    model_flux_npy='retrieval_model_flux_N2.npy',
)


In [ ]:
plot_pt_profile_piette(
    workpath=workpath,
    retrieval_id=retrieval_id,
    param_keys=_PIETTE_EQUIL_PARAMS,
)


In [ ]:
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
    data_npy='extracted_spectra_combined_two_nights.npy',
    err_npy='extracted_spectra_combined_two_nights_err.npy',
    data_dir=workpath / 'combined_two_nights',
)


### Analysis: 2768012 (Piette P-T) vs DG runs — P-T model comparison

#### Summary comparison (posterior mean ± σ)

| Parameter | 2734585 (DG, mode A) | 2743787 (DG, mode B) | **2768012 (Piette)** | Comment |
|---|---|---|---|---|
| lnZ | −14 854 | −14 868 | **−14 843** | **Piette preferred by Δ = +11–25 nats** |
| C/O | 0.463 ± 0.009 | 0.514 ± 0.010 | **0.494 ± 0.010** | Near-solar across all runs |
| [C/H] | −0.169 ± 0.040 | −0.437 ± 0.044 | **−0.275 ± 0.030** | Piette intermediate |
| log ¹²CO/¹³CO | 1.004 ± 0.040 | 1.030 ± 0.042 | **1.000 ± 0.055** | ~10 across all |
| log_g | 4.652 ± 0.064 | 3.943 ± 0.029 | **4.294 ± 0.040** | Piette between the two DG modes |
| T_anchor (1 bar) | — | — | **2110 ± 23 K** | Well-constrained photospheric anchor |
| vsini | 6.82 ± 0.14 | 7.05 ± 0.22 | **7.17 ± 0.17** | Consistent |
| χ² (N1/N2) | 1.47 / 1.36 | 1.47 / 1.36 | **1.47 / 1.36** | Identical per-pixel fit |
| Posterior samples | 4 112 | 3 348 | **4 318** | All N=300 |

#### Finding 1 — Piette P-T is preferred by Bayesian evidence

Piette P-T (lnZ = −14 843) is preferred over the DG runs by Δ lnZ = +11 to +25 nats,
constituting 'decisive' Bayesian evidence (Jeffreys scale: Δ lnZ > 5 = decisive).
The Piette parameterisation (8 nodes, PCHIP spline, 3 fewer free parameters than DG)
provides a better fit with less prior volume — a genuine model improvement.

#### Finding 2 — Piette avoids the DG log_g bimodality

The DG model has two modes at N=300: log_g ≈ 3.94 (mode B) and log_g ≈ 4.65 (mode A),
separated by ~23σ. The Piette model converges to an intermediate log_g = 4.29, without
evidence of bimodality. This is likely because the Piette parameterisation constrains the
T(P) profile more tightly near the photosphere (T_anchor at 1 bar is directly retrieved),
reducing the log_g–T_bottom degeneracy that drives the DG bimodality.

#### Finding 3 — Chemistry consistent across models (near-solar, ~10× sub-solar ¹²C/¹³C)

C/O = 0.49–0.51 (near solar 0.55), [C/H] = −0.17 to −0.44 (sub-solar),
log ¹²CO/¹³CO ≈ 1.0 (ratio ≈ 10) across all runs.
The isotopologue ratio is robustly sub-solar (solar = 70; 10 = very enriched in ¹³C).

#### Next step

Re-run Piette P-T (003PM-EQ) at N=700 with blaze-corrected sigmaclipper data to obtain
well-sampled posteriors. The current N=300 run (4318 samples) is marginally sufficient
for 16 dimensions but a factor-of-2 increase in live points would substantially improve
parameter uncertainties.


## Retrieval: 2819945

In [ ]:
import sys
sys.path.insert(0, '/data2/peng/Recipe_DH_Tau_B')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from analysis import (
    plot_ccf, plot_equil_chemistry,
    plot_binned_model_vs_data,
    _PIETTE_EQUIL_PARAMS,
    plot_pt_profile_piette,
)
from tasting_analysis import run_ccf_workflow, run_equil_chemistry_workflow

workpath     = Path('/data2/peng')
retrieval_id = '2819945_N300_ev0.5_Normmedian_PerChipScaleFalse'
night        = '2022-12-31'

print('retrieval exists:', (workpath / 'retrievals' / retrieval_id).exists())

ccf_out = run_ccf_workflow(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night=night,
    rvlag=np.arange(-10, 10, 0.1),
    clean_grids=((-10, -1), (0, 10)),
    n_shuffle=200,
    random_seed=42,
    combined_spectrum_name='extracted_spectra_combined_sigmaclipper.npy',
)
print(f"Peak SNR = {ccf_out['peak_snr']:.2f} at RV = {ccf_out['peak_rv']:.1f} km/s")
print(f"z-score = {ccf_out['z_score']:.2f}")
plot_ccf(ccf_out, retrieval_id)

chem_out = run_equil_chemistry_workflow(workpath=workpath, retrieval_id=retrieval_id)
print('Posterior shape:', chem_out['posterior'].shape)
display(chem_out['summary'])
print('\nText summary:')
for _, row in chem_out['summary'].iterrows():
    print(
        f"{row['parameter']}: median={row['p50']:.4g}, "
        f"1σ=[-{row['minus_1sigma']:.3g}, +{row['plus_1sigma']:.3g}]"
    )
plot_equil_chemistry(chem_out, retrieval_id)


In [ ]:
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)

plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2023-01-01',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)

plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
    data_npy='extracted_spectra_combined_two_nights.npy',
    err_npy='extracted_spectra_combined_two_nights_err.npy',
    data_dir=workpath / 'combined_two_nights',
)



In [ ]:
plot_pt_profile_piette(
    workpath=workpath,
    retrieval_id=retrieval_id,
    param_keys=_PIETTE_EQUIL_PARAMS,
)

## Retrieval: 2898115

In [ ]:
import sys
sys.path.insert(0, '/data2/peng/Recipe_DH_Tau_B')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from analysis import (
    plot_ccf, plot_equil_chemistry,
    plot_binned_model_vs_data,
    _PIETTE_EQUIL_PARAMS,
    plot_pt_profile_piette,
)
from tasting_analysis import run_ccf_workflow, run_equil_chemistry_workflow

workpath     = Path('/data2/peng')
retrieval_id = '2898115_N600_ev0.5_Normmedian_PerChipScaleFalse'
night        = '2022-12-31'

print('retrieval exists:', (workpath / 'retrievals' / retrieval_id).exists())

ccf_out = run_ccf_workflow(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night=night,
    rvlag=np.arange(-10, 10, 0.1),
    clean_grids=((-10, -1), (0, 10)),
    n_shuffle=200,
    random_seed=42,
    combined_spectrum_name='extracted_spectra_combined_sigmaclipper.npy',
)
print(f"Peak SNR = {ccf_out['peak_snr']:.2f} at RV = {ccf_out['peak_rv']:.1f} km/s")
print(f"z-score = {ccf_out['z_score']:.2f}")
plot_ccf(ccf_out, retrieval_id)

chem_out = run_equil_chemistry_workflow(workpath=workpath, retrieval_id=retrieval_id)
print('Posterior shape:', chem_out['posterior'].shape)
display(chem_out['summary'])
print('\nText summary:')
for _, row in chem_out['summary'].iterrows():
    print(
        f"{row['parameter']}: median={row['p50']:.4g}, "
        f"1σ=[-{row['minus_1sigma']:.3g}, +{row['plus_1sigma']:.3g}]"
    )
plot_equil_chemistry(chem_out, retrieval_id)


In [ ]:
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)

plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2023-01-01',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)

plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
    data_npy='extracted_spectra_combined_two_nights.npy',
    err_npy='extracted_spectra_combined_two_nights_err.npy',
    data_dir=workpath / 'combined_two_nights',
)


In [ ]:
plot_pt_profile_piette(
    workpath=workpath,
    retrieval_id=retrieval_id,
    param_keys=_PIETTE_EQUIL_PARAMS,
)


## Retrieval: 3078846

In [ ]:
import sys
sys.path.insert(0, '/data2/peng/Recipe_DH_Tau_B')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from analysis import (
    plot_ccf, plot_equil_chemistry,
    plot_binned_model_vs_data,
    _PIETTE_EQUIL_PARAMS,
    plot_pt_profile_piette,
)
from tasting_analysis import run_ccf_workflow, run_equil_chemistry_workflow

workpath     = Path('/data2/peng')
retrieval_id = '3078846_N600_ev0.5_Normmedian_PerChipScaleFalse'
night        = '2022-12-31'

print('retrieval exists:', (workpath / 'retrievals' / retrieval_id).exists())

ccf_out = run_ccf_workflow(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night=night,
    rvlag=np.arange(-10, 10, 0.1),
    clean_grids=((-10, -1), (0, 10)),
    n_shuffle=200,
    random_seed=42,
    combined_spectrum_name='extracted_spectra_combined_flux_cal.npy',
)
print(f"Peak SNR = {ccf_out['peak_snr']:.2f} at RV = {ccf_out['peak_rv']:.1f} km/s")
print(f"z-score = {ccf_out['z_score']:.2f}")
plot_ccf(ccf_out, retrieval_id)

chem_out = run_equil_chemistry_workflow(workpath=workpath, retrieval_id=retrieval_id)
print('Posterior shape:', chem_out['posterior'].shape)
display(chem_out['summary'])
print('\nText summary:')
for _, row in chem_out['summary'].iterrows():
    print(
        f"{row['parameter']}: median={row['p50']:.4g}, "
        f"1σ=[-{row['minus_1sigma']:.3g}, +{row['plus_1sigma']:.3g}]"
    )
plot_equil_chemistry(chem_out, retrieval_id)


In [ ]:
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)

plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2023-01-01',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)

plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
    data_npy='extracted_spectra_combined_two_nights.npy',
    err_npy='extracted_spectra_combined_two_nights_err.npy',
    data_dir=workpath / 'combined_two_nights',
)


In [ ]:
plot_pt_profile_piette(
    workpath=workpath,
    retrieval_id=retrieval_id,
    param_keys=_PIETTE_EQUIL_PARAMS,
)


## Retrieval: 3176730

In [ ]:
import sys
sys.path.insert(0, '/data2/peng/Recipe_DH_Tau_B')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from analysis import (
    plot_ccf, plot_equil_chemistry,
    plot_binned_model_vs_data,
    _PIETTE_EQUIL_PARAMS_MR,
    plot_pt_profile_piette,
)
from tasting_analysis import run_ccf_workflow, run_equil_chemistry_workflow

workpath     = Path('/data2/peng')
retrieval_id = '3176730_N600_ev0.5_Normmedian_PerChipScaleFalse'
night        = '2022-12-31'

print('retrieval exists:', (workpath / 'retrievals' / retrieval_id).exists())

ccf_out = run_ccf_workflow(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night=night,
    rvlag=np.arange(-10, 10, 0.1),
    clean_grids=((-10, -1), (0, 10)),
    n_shuffle=200,
    random_seed=42,
    combined_spectrum_name='extracted_spectra_combined_flux_cal.npy',
)
print(f"Peak SNR = {ccf_out['peak_snr']:.2f} at RV = {ccf_out['peak_rv']:.1f} km/s")
print(f"z-score = {ccf_out['z_score']:.2f}")
plot_ccf(ccf_out, retrieval_id)

chem_out = run_equil_chemistry_workflow(workpath=workpath, retrieval_id=retrieval_id)
print('Posterior shape:', chem_out['posterior'].shape)
display(chem_out['summary'])
print('Text summary:')
for _, row in chem_out['summary'].iterrows():
    print(
        f"{row['parameter']}: median={row['p50']:.4g}, "
        f"1σ=[-{row['minus_1sigma']:.3g}, +{row['plus_1sigma']:.3g}]"
    )
plot_equil_chemistry(chem_out, retrieval_id)

In [ ]:
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)

plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2023-01-01',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)

plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
    data_npy='extracted_spectra_combined_two_nights.npy',
    err_npy='extracted_spectra_combined_two_nights_err.npy',
    data_dir=workpath / 'combined_two_nights',
)

In [ ]:
plot_pt_profile_piette(
    workpath=workpath,
    retrieval_id=retrieval_id,
    param_keys=_PIETTE_EQUIL_PARAMS_MR,
)

In [ ]:
# Derived log_g posterior from log_M and log_R samples.
# log_g (cgs) = log10( G * M / R^2 )  where G = 6.674e-8 cgs
# log_M in log10(M_Jup), log_R in log10(R_Jup)
# M_Jup = 1.898e30 g, R_Jup = 7.149e9 cm

posterior = np.load(workpath / 'retrievals' / retrieval_id / 'final_posterior.npy')
col_mr = {k: i for i, k in enumerate(_PIETTE_EQUIL_PARAMS_MR)}

log_M_samp = posterior[:, col_mr['log_M']]
log_R_samp = posterior[:, col_mr['log_R']]

G_cgs   = 6.674e-8
M_JUP_G = 1.898e30
R_JUP_CM = 7.149e9

M_g  = 10**log_M_samp * M_JUP_G
R_cm = 10**log_R_samp * R_JUP_CM
log_g_derived = np.log10(G_cgs * M_g / R_cm**2)

p16, p50, p84 = np.percentile(log_g_derived, [16, 50, 84])
print(f'log_g (derived): {p50:.3f} +{p84-p50:.3f} / -{p50-p16:.3f}')
print(f'  → g = {10**p50:.0f} cm/s²  (Xuan+2024 DH Tau b expectation: ~3.64 dex)')

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(log_g_derived, bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
ax.axvline(p50, color='navy',   linewidth=1.5, label=f'median = {p50:.3f}')
ax.axvline(p16, color='navy',   linewidth=1.0, linestyle='--')
ax.axvline(p84, color='navy',   linewidth=1.0, linestyle='--')
#ax.axvline(3.643, color='tomato', linewidth=1.5, linestyle=':',
           #label='Xuan+2024 expectation (3.643)')
ax.set_xlabel(r'$\log g$ (cgs)', fontsize=12)
ax.set_ylabel('Posterior samples', fontsize=12)
ax.set_title(f'{retrieval_id} Derived $\log g$ from $\log M$ + $\log R$ posteriors')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 3246017

In [ ]:
import sys
sys.path.insert(0, '/data2/peng/Recipe_DH_Tau_B')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from analysis import (
    plot_ccf, plot_equil_chemistry,
    plot_binned_model_vs_data,
    _PIETTE_EQUIL_PARAMS_MR,
    plot_pt_profile_piette,
)
from tasting_analysis import run_ccf_workflow, run_equil_chemistry_workflow

workpath     = Path('/data2/peng')
retrieval_id = '3246017_N800_ev0.5_Normmedian_PerChipScaleFalse'
night        = '2022-12-31'

print('retrieval exists:', (workpath / 'retrievals' / retrieval_id).exists())

ccf_out = run_ccf_workflow(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night=night,
    rvlag=np.arange(-10, 10, 0.1),
    clean_grids=((-10, -1), (0, 10)),
    n_shuffle=200,
    random_seed=42,
    combined_spectrum_name='extracted_spectra_combined_flux_cal.npy',
)
print(f"Peak SNR = {ccf_out['peak_snr']:.2f} at RV = {ccf_out['peak_rv']:.1f} km/s")
print(f"z-score = {ccf_out['z_score']:.2f}")
plot_ccf(ccf_out, retrieval_id)

chem_out = run_equil_chemistry_workflow(workpath=workpath, retrieval_id=retrieval_id)
print('Posterior shape:', chem_out['posterior'].shape)
display(chem_out['summary'])
print('Text summary:')
for _, row in chem_out['summary'].iterrows():
    print(
        f"{row['parameter']}: median={row['p50']:.4g}, "
        f"1σ=[-{row['minus_1sigma']:.3g}, +{row['plus_1sigma']:.3g}]"
    )
plot_equil_chemistry(chem_out, retrieval_id)

# Derived log_g posterior from log_M and log_R samples.
# log_g (cgs) = log10( G * M / R^2 )  where G = 6.674e-8 cgs
# log_M in log10(M_Jup), log_R in log10(R_Jup)
# M_Jup = 1.898e30 g, R_Jup = 7.149e9 cm

posterior = np.load(workpath / 'retrievals' / retrieval_id / 'final_posterior.npy')
col_mr = {k: i for i, k in enumerate(_PIETTE_EQUIL_PARAMS_MR)}

log_M_samp = posterior[:, col_mr['log_M']]
log_R_samp = posterior[:, col_mr['log_R']]

G_cgs   = 6.674e-8
M_JUP_G = 1.898e30
R_JUP_CM = 7.149e9

M_g  = 10**log_M_samp * M_JUP_G
R_cm = 10**log_R_samp * R_JUP_CM
log_g_derived = np.log10(G_cgs * M_g / R_cm**2)

p16, p50, p84 = np.percentile(log_g_derived, [16, 50, 84])
print(f'log_g (derived): {p50:.3f} +{p84-p50:.3f} / -{p50-p16:.3f}')
print(f'  → g = {10**p50:.0f} cm/s²  (Xuan+2024 DH Tau b expectation: ~3.64 dex)')

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(log_g_derived, bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
ax.axvline(p50, color='navy',   linewidth=1.5, label=f'median = {p50:.3f}')
ax.axvline(p16, color='navy',   linewidth=1.0, linestyle='--')
ax.axvline(p84, color='navy',   linewidth=1.0, linestyle='--')
#ax.axvline(3.643, color='tomato', linewidth=1.5, linestyle=':',
           #label='Xuan+2024 expectation (3.643)')
ax.set_xlabel(r'$\log g$ (cgs)', fontsize=12)
ax.set_ylabel('Posterior samples', fontsize=12)
ax.set_title(f'{retrieval_id} Derived $\log g$ from $\log M$ + $\log R$ posteriors')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)

plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2023-01-01',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)

plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
    data_npy='extracted_spectra_combined_two_nights.npy',
    err_npy='extracted_spectra_combined_two_nights_err.npy',
    data_dir=workpath / 'combined_two_nights',
)



In [ ]:
plot_pt_profile_piette(
    workpath=workpath,
    retrieval_id=retrieval_id,
    param_keys=_PIETTE_EQUIL_PARAMS_MR,
)

## 4151825

In [ ]:
import sys
sys.path.insert(0, '/data2/peng/Recipe_DH_Tau_B')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from analysis import (
    plot_ccf, plot_equil_chemistry,
    plot_binned_model_vs_data,
    _PIETTE_EQUIL_PARAMS_MR,
    plot_pt_profile_piette,
)
from tasting_analysis import run_ccf_workflow, run_equil_chemistry_workflow

workpath     = Path('/data2/peng')
retrieval_id = '4151825_N800_ev0.5_NormNone_PerChipScaleFalse'
night        = '2022-12-31'

print('retrieval exists:', (workpath / 'retrievals' / retrieval_id).exists())

ccf_out = run_ccf_workflow(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night=night,
    rvlag=np.arange(-10, 10, 0.1),
    clean_grids=((-10, -1), (0, 10)),
    n_shuffle=200,
    random_seed=42,
    combined_spectrum_name='extracted_spectra_combined_flux_cal.npy',
)
print(f"Peak SNR = {ccf_out['peak_snr']:.2f} at RV = {ccf_out['peak_rv']:.1f} km/s")
print(f"z-score = {ccf_out['z_score']:.2f}")
plot_ccf(ccf_out, retrieval_id)

chem_out = run_equil_chemistry_workflow(workpath=workpath, retrieval_id=retrieval_id)
print('Posterior shape:', chem_out['posterior'].shape)
display(chem_out['summary'])
print('Text summary:')
for _, row in chem_out['summary'].iterrows():
    print(
        f"{row['parameter']}: median={row['p50']:.4g}, "
        f"1σ=[-{row['minus_1sigma']:.3g}, +{row['plus_1sigma']:.3g}]"
    )
plot_equil_chemistry(chem_out, retrieval_id)

# Derived log_g posterior from log_M and log_R samples.
# log_g (cgs) = log10( G * M / R^2 )  where G = 6.674e-8 cgs
# log_M in log10(M_Jup), log_R in log10(R_Jup)
# M_Jup = 1.898e30 g, R_Jup = 7.149e9 cm

posterior = np.load(workpath / 'retrievals' / retrieval_id / 'final_posterior.npy')
col_mr = {k: i for i, k in enumerate(_PIETTE_EQUIL_PARAMS_MR)}

log_M_samp = posterior[:, col_mr['log_M']]
log_R_samp = posterior[:, col_mr['log_R']]

G_cgs   = 6.674e-8
M_JUP_G = 1.898e30
R_JUP_CM = 7.149e9

M_g  = 10**log_M_samp * M_JUP_G
R_cm = 10**log_R_samp * R_JUP_CM
log_g_derived = np.log10(G_cgs * M_g / R_cm**2)

p16, p50, p84 = np.percentile(log_g_derived, [16, 50, 84])
print(f'log_g (derived): {p50:.3f} +{p84-p50:.3f} / -{p50-p16:.3f}')
print(f'  → g = {10**p50:.0f} cm/s²  (Xuan+2024 DH Tau b expectation: ~3.64 dex)')

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(log_g_derived, bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
ax.axvline(p50, color='navy',   linewidth=1.5, label=f'median = {p50:.3f}')
ax.axvline(p16, color='navy',   linewidth=1.0, linestyle='--')
ax.axvline(p84, color='navy',   linewidth=1.0, linestyle='--')
#ax.axvline(3.643, color='tomato', linewidth=1.5, linestyle=':',
           #label='Xuan+2024 expectation (3.643)')
ax.set_xlabel(r'$\log g$ (cgs)', fontsize=12)
ax.set_ylabel('Posterior samples', fontsize=12)
ax.set_title(f'{retrieval_id} Derived $\log g$ from $\log M$ + $\log R$ posteriors')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

#plot alos M and R posteriors converted into linear space, and show the median and 1σ intervals.
p16_M, p50_M, p84_M = np.percentile(M_g/M_JUP_G, [16, 50, 84])
p16_R, p50_R, p84_R = np.percentile(R_cm/R_JUP_CM, [16, 50, 84])


fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(M_g/M_JUP_G, bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
ax[1].hist(R_cm/R_JUP_CM, bins=60, color='steelblue', edgecolor='white', linewidth=0.3)

ax[0].axvline(p50_M, color='navy',   linewidth=1.5, label=f'median = {p50_M:.3f}')
ax[0].axvline(p16_M, color='navy',   linewidth=1.0, linestyle='--')
ax[0].axvline(p84_M, color='navy',   linewidth=1.0, linestyle='--')
ax[1].axvline(p50_R, color='navy',   linewidth=1.5, label=f'median = {p50_R:.3f}')
ax[1].axvline(p16_R, color='navy',   linewidth=1.0, linestyle='--')
ax[1].axvline(p84_R, color='navy',   linewidth=1.0, linestyle='--')

ax[0].set_xlabel(r'$M$ ($M_{Jup}$)', fontsize=12)
ax[1].set_xlabel(r'$R$ ($R_{Jup}$)', fontsize=12)
ax[0].set_ylabel('Posterior samples', fontsize=12)

ax[0].set_title(f'{retrieval_id} $\log M$ posterior', fontsize=12)
ax[1].set_title(f'{retrieval_id} $\log R$ posterior', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)

plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2023-01-01',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)

plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
    data_npy='extracted_spectra_combined_two_nights.npy',
    err_npy='extracted_spectra_combined_two_nights_err.npy',
    data_dir=workpath / 'combined_two_nights',
)



In [ ]:
plot_pt_profile_piette(
    workpath=workpath,
    retrieval_id=retrieval_id,
    param_keys=_PIETTE_EQUIL_PARAMS_MR,
)

## 628541

In [ ]:
import sys
sys.path.insert(0, '/data2/peng/Recipe_DH_Tau_B')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from analysis import (
    plot_ccf, plot_equil_chemistry,
    plot_binned_model_vs_data,
    _PIETTE_EQUIL_PARAMS_MR,
    plot_pt_profile_piette,
)
from tasting_analysis import run_ccf_workflow, run_equil_chemistry_workflow

workpath     = Path('/data2/peng')
retrieval_id = '628541_N800_ev0.5_NormNone_PerChipScaleFalse'
night        = '2022-12-31'

print('retrieval exists:', (workpath / 'retrievals' / retrieval_id).exists())

ccf_out = run_ccf_workflow(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night=night,
    rvlag=np.arange(-10, 10, 0.1),
    clean_grids=((-10, -1), (0, 10)),
    n_shuffle=200,
    random_seed=42,
    combined_spectrum_name='extracted_spectra_combined_flux_cal.npy',
)
print(f"Peak SNR = {ccf_out['peak_snr']:.2f} at RV = {ccf_out['peak_rv']:.1f} km/s")
print(f"z-score = {ccf_out['z_score']:.2f}")
plot_ccf(ccf_out, retrieval_id)

chem_out = run_equil_chemistry_workflow(workpath=workpath, retrieval_id=retrieval_id)
print('Posterior shape:', chem_out['posterior'].shape)
display(chem_out['summary'])
print('Text summary:')
for _, row in chem_out['summary'].iterrows():
    print(
        f"{row['parameter']}: median={row['p50']:.4g}, "
        f"1σ=[-{row['minus_1sigma']:.3g}, +{row['plus_1sigma']:.3g}]"
    )
plot_equil_chemistry(chem_out, retrieval_id)

# Derived log_g posterior from log_M and log_R samples.
# log_g (cgs) = log10( G * M / R^2 )  where G = 6.674e-8 cgs
# log_M in log10(M_Jup), log_R in log10(R_Jup)
# M_Jup = 1.898e30 g, R_Jup = 7.149e9 cm

posterior = np.load(workpath / 'retrievals' / retrieval_id / 'final_posterior.npy')
col_mr = {k: i for i, k in enumerate(_PIETTE_EQUIL_PARAMS_MR)}

log_M_samp = posterior[:, col_mr['log_M']]
log_R_samp = posterior[:, col_mr['log_R']]

G_cgs   = 6.674e-8
M_JUP_G = 1.898e30
R_JUP_CM = 7.149e9

M_g  = 10**log_M_samp * M_JUP_G
R_cm = 10**log_R_samp * R_JUP_CM
log_g_derived = np.log10(G_cgs * M_g / R_cm**2)

p16, p50, p84 = np.percentile(log_g_derived, [16, 50, 84])
print(f'log_g (derived): {p50:.3f} +{p84-p50:.3f} / -{p50-p16:.3f}')
print(f'  → g = {10**p50:.0f} cm/s²  (Xuan+2024 DH Tau b expectation: ~3.64 dex)')

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(log_g_derived, bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
ax.axvline(p50, color='navy',   linewidth=1.5, label=f'median = {p50:.3f}')
ax.axvline(p16, color='navy',   linewidth=1.0, linestyle='--')
ax.axvline(p84, color='navy',   linewidth=1.0, linestyle='--')
#ax.axvline(3.643, color='tomato', linewidth=1.5, linestyle=':',
           #label='Xuan+2024 expectation (3.643)')
ax.set_xlabel(r'$\log g$ (cgs)', fontsize=12)
ax.set_ylabel('Posterior samples', fontsize=12)
ax.set_title(f'{retrieval_id} Derived $\log g$ from $\log M$ + $\log R$ posteriors')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

#plot alos M and R posteriors converted into linear space, and show the median and 1σ intervals.
p16_M, p50_M, p84_M = np.percentile(M_g/M_JUP_G, [16, 50, 84])
p16_R, p50_R, p84_R = np.percentile(R_cm/R_JUP_CM, [16, 50, 84])


fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(M_g/M_JUP_G, bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
ax[1].hist(R_cm/R_JUP_CM, bins=60, color='steelblue', edgecolor='white', linewidth=0.3)

ax[0].axvline(p50_M, color='navy',   linewidth=1.5, label=f'median = {p50_M:.3f}')
ax[0].axvline(p16_M, color='navy',   linewidth=1.0, linestyle='--')
ax[0].axvline(p84_M, color='navy',   linewidth=1.0, linestyle='--')
ax[1].axvline(p50_R, color='navy',   linewidth=1.5, label=f'median = {p50_R:.3f}')
ax[1].axvline(p16_R, color='navy',   linewidth=1.0, linestyle='--')
ax[1].axvline(p84_R, color='navy',   linewidth=1.0, linestyle='--')

ax[0].set_xlabel(r'$M$ ($M_{Jup}$)', fontsize=12)
ax[1].set_xlabel(r'$R$ ($R_{Jup}$)', fontsize=12)
ax[0].set_ylabel('Posterior samples', fontsize=12)

ax[0].set_title(f'{retrieval_id} $\log M$ posterior', fontsize=12)
ax[1].set_title(f'{retrieval_id} $\log R$ posterior', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)

plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2023-01-01',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
)

plot_binned_model_vs_data(
    workpath=workpath,
    retrieval_id=retrieval_id,
    night='2022-12-31',
    model_wave_npy='retrieval_model_wave.npy',
    model_flux_npy='retrieval_model_flux_scaled.npy',
    data_npy='extracted_spectra_combined_two_nights.npy',
    err_npy='extracted_spectra_combined_two_nights_err.npy',
    data_dir=workpath / 'combined_two_nights',
)



In [ ]:
plot_pt_profile_piette(
    workpath=workpath,
    retrieval_id=retrieval_id,
    param_keys=_PIETTE_EQUIL_PARAMS_MR,
)